#MUTLI-LAYER TF-BERT for classify

In [1]:
import torch
import torch.nn as nn
from transformers import BertModel, BertTokenizer
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import os
import random

np.random.seed(42)
random.seed(42)
torch.manual_seed(42)



In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs('plots', exist_ok=True)



In [ ]:
visual_features = np.load("/kaggle/input/multi-final-cmu-pro/visual_features.npy")
audio_features = np.load("/kaggle/input/multi-final-cmu-pro/audio_features.npy")
labels_df = pd.read_csv("/kaggle/input/multi-final-cmu-pro/labels.csv")

def split_data(mode):
    idx = labels_df[labels_df["mode"] == mode].index
    return {
        "text": labels_df.loc[idx, "text"].values,
        "visual": visual_features[idx],
        "audio": audio_features[idx],
        "annotation": labels_df.loc[idx, "annotation"].map({"Positive": 2, "Neutral": 1, "Negative": 0}).values
    }

train_data = split_data("train")
test_data = split_data("test")
valid_data = split_data("valid")



In [ ]:

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Dataset and Dataloader
class MultimodalDataset(Dataset):
    def __init__(self, data):
        self.text = data["text"]
        self.visual = torch.FloatTensor(data["visual"])
        self.audio = torch.FloatTensor(data["audio"])
        self.annotation = torch.LongTensor(data["annotation"])

    def __len__(self):
        return len(self.text)

    def __getitem__(self, idx):
        text_encoded = tokenizer(
            self.text[idx],
            padding="max_length",
            truncation=True,
            max_length=50,
            return_tensors="pt"
        )
        return {
            "input_ids": text_encoded["input_ids"].squeeze(0),
            "attention_mask": text_encoded["attention_mask"].squeeze(0),
            "visual": self.visual[idx],
            "audio": self.audio[idx],
            "annotation": self.annotation[idx]
        }

batch_size = 32
train_loader = DataLoader(MultimodalDataset(train_data), batch_size=batch_size, shuffle=True)
test_loader = DataLoader(MultimodalDataset(test_data), batch_size=batch_size)
valid_loader = DataLoader(MultimodalDataset(valid_data), batch_size=batch_size)



In [ ]:
# 3. Model Architecture
class TCT(nn.Module):
    def __init__(self, text_dim=768, audio_dim=74, visual_dim=128, num_heads=2):
        super().__init__()
        self.audio_proj = nn.Linear(audio_dim, text_dim)
        self.visual_proj = nn.Linear(visual_dim, text_dim)
        self.multihead_attn = nn.MultiheadAttention(text_dim, num_heads)
        self.layer_norm1 = nn.LayerNorm(text_dim)
        self.layer_norm2 = nn.LayerNorm(text_dim)
        self.ffn = nn.Sequential(
            nn.Linear(text_dim, 4 * text_dim),
            nn.ReLU(),
            nn.Linear(4 * text_dim, text_dim)
        )

    def forward(self, text, audio, visual):
        audio_proj = self.audio_proj(audio).unsqueeze(1)
        visual_proj = self.visual_proj(visual).unsqueeze(1)

        Q = text.transpose(0, 1)
        K = audio_proj.transpose(0, 1)
        V = visual_proj.transpose(0, 1)

        attn_output, _ = self.multihead_attn(Q, K, V)
        attn_output = attn_output.transpose(0, 1)

        output = self.layer_norm1(text + attn_output)
        ffn_output = self.ffn(output)
        output = self.layer_norm2(output + ffn_output)
        return output



In [ ]:
class TF_BERT_Classifier(nn.Module):
    def __init__(self, tcf_start_layer=6):
        super().__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        self.tct = TCT()
        self.classifier = nn.Linear(768, 3)  # 3 classes
        self.tcf_start_layer = tcf_start_layer

    def forward(self, input_ids, attention_mask, visual, audio):
        attention_mask = attention_mask.float().unsqueeze(1).unsqueeze(2)
        text_output = self.bert.embeddings(input_ids)

        for i, layer in enumerate(self.bert.encoder.layer):
            layer_output = layer(text_output, attention_mask=attention_mask)[0]
            text_output = layer_output

            if i >= self.tcf_start_layer:
                text_output = self.tct(text_output, audio, visual)

        pooled_output = text_output.mean(dim=1)
        logits = self.classifier(pooled_output)
        return logits



In [ ]:
# 4. Training Setup
model = TF_BERT_Classifier(tcf_start_layer=6).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
criterion = nn.CrossEntropyLoss()


patience = 5
best_acc2 = 0.0
early_stop_counter = 0



In [ ]:
# Training
def train_epoch(model, dataloader):
    model.train()
    total_loss = 0
    progress_bar = tqdm(dataloader, desc="Training")
    for batch in progress_bar:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        visual = batch["visual"].to(device)
        audio = batch["audio"].to(device)
        annotation = batch["annotation"].to(device)

        logits = model(input_ids, attention_mask, visual, audio)
        loss = criterion(logits, annotation)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        progress_bar.set_postfix({"loss": loss.item()})
    return total_loss / len(dataloader)



In [ ]:
def validate(model, dataloader):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Validation"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            visual = batch["visual"].to(device)
            audio = batch["audio"].to(device)
            annotation = batch["annotation"].to(device)

            logits = model(input_ids, attention_mask, visual, audio)
            loss = criterion(logits, annotation)
            total_loss += loss.item()
    return total_loss / len(dataloader)



In [ ]:
def evaluate(model, dataloader, return_confusion=False):
    model.eval()
    preds, anns = [], []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluation"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            visual = batch["visual"].to(device)
            audio = batch["audio"].to(device)
            annotation = batch["annotation"].to(device)

            logits = model(input_ids, attention_mask, visual, audio)
            preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            anns.extend(annotation.cpu().numpy())


    acc = accuracy_score(anns, preds)
    f1 = f1_score(anns, preds, average='weighted')


    binary_anns = [0 if a == 0 else 1 for a in anns]
    binary_preds = [0 if p == 0 else 1 for p in preds]
    acc2 = accuracy_score(binary_anns, binary_preds)
    binary_f1 = f1_score(binary_anns, binary_preds, average='weighted')

    if return_confusion:

        plt.figure(figsize=(10, 8))
        cm = confusion_matrix(anns, preds)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=['Negative', 'Neutral', 'Positive'],
                    yticklabels=['Negative', 'Neutral', 'Positive'])
        plt.title('Confusion Matrix (All Classes)')
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
        plt.savefig('plots/confusion_matrix_all.png')
        plt.close()


        binary_cm = confusion_matrix(binary_anns, binary_preds)
        plt.figure(figsize=(8, 6))
        sns.heatmap(binary_cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=['Negative', 'Non-Negative'],
                    yticklabels=['Negative', 'Non-Negative'])
        plt.title('Confusion Matrix (Negative vs Non-Negative)')
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
        plt.savefig('plots/confusion_matrix_binary.png')
        plt.close()

    return acc, f1, acc2, binary_f1



In [ ]:
# 6. Training Loop
max_epochs = 20
train_losses = []
val_losses = []
val_metrics = {'acc': [], 'f1': [], 'acc2': [], 'binary_f1': []}

for epoch in range(max_epochs):

    epoch_train_loss = train_epoch(model, train_loader)
    train_losses.append(epoch_train_loss)


    epoch_val_loss = validate(model, valid_loader)
    val_losses.append(epoch_val_loss)


    acc, f1, acc2, binary_f1 = evaluate(model, valid_loader)
    val_metrics['acc'].append(acc)
    val_metrics['f1'].append(f1)
    val_metrics['acc2'].append(acc2)
    val_metrics['binary_f1'].append(binary_f1)

    print(f"Epoch {epoch+1}/{max_epochs}: Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | "
          f"Val Acc: {acc:.3f} | Val F1: {f1:.3f} | Val ACC2: {acc2:.3f} | Val Binary F1: {binary_f1:.3f}")


    if acc2 > best_acc2:
        best_acc2 = acc2
        early_stop_counter = 0
        torch.save(model.state_dict(), 'best_model.pt')
        print(f"New best ACC2: {best_acc2:.4f} - Saving model")
    else:
        early_stop_counter += 1
        print(f"ACC2 did not improve. Early stopping counter: {early_stop_counter}/{patience}")
        if early_stop_counter >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs!")
            break



In [ ]:

model.load_state_dict(torch.load('best_model.pt'))
print(f"Loaded best model with ACC2: {best_acc2:.4f}")



In [ ]:

plt.figure(figsize=(15, 10))

# Loss curves
plt.subplot(2, 2, 1)
plt.plot(train_losses, label='Training Loss', color='blue')
plt.plot(val_losses, label='Validation Loss', color='orange')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()

# Accuracy metrics with ACC2 highlighted
plt.subplot(2, 2, 2)
plt.plot(val_metrics['acc'], label='Accuracy (3-class)', color='green', alpha=0.7)
plt.plot(val_metrics['acc2'], label='ACC2 (Negative vs Non-Negative)', color='red', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Score')
plt.title('Accuracy Metrics (Early Stopping on ACC2)')
plt.legend()

# F1 scores
plt.subplot(2, 2, 3)
plt.plot(val_metrics['f1'], label='F1 Score (3-class)', color='purple')
plt.plot(val_metrics['binary_f1'], label='Binary F1 (Negative vs Non-Negative)', color='brown')
plt.xlabel('Epoch')
plt.ylabel('Score')
plt.title('F1 Scores')
plt.legend()

# Highlight ACC2 alone
plt.subplot(2, 2, 4)
plt.plot(val_metrics['acc2'], label='ACC2 (Negative vs Non-Negative)', color='red', marker='o')
plt.axhline(y=best_acc2, color='black', linestyle='--', alpha=0.7, label=f'Best ACC2: {best_acc2:.3f}')
plt.xlabel('Epoch')
plt.ylabel('Score')
plt.title('ACC2 Progression (Early Stopping Metric)')
plt.legend()

plt.tight_layout()
plt.savefig('plots/training_metrics.png', dpi=300, bbox_inches='tight')
plt.close()



In [ ]:
test_acc, test_f1, test_acc2, test_binary_f1 = evaluate(model, test_loader, return_confusion=True)
print(f"\nTest Results:")
print(f"Accuracy (All Classes): {test_acc:.3f}")
print(f"F1 Score (Weighted): {test_f1:.3f}")
print(f"ACC2 (Negative vs Non-Negative): {test_acc2:.3f}")
print(f"Binary F1 (Negative vs Non-Negative): {test_binary_f1:.3f}")

# Save all metrics to CSV
results_df = pd.DataFrame({
    'Metric': ['Accuracy', 'F1 Score', 'ACC2 (Negative vs Non-Negative)', 'Binary F1'],
    'Value': [test_acc, test_f1, test_acc2, test_binary_f1]
})
results_df.to_csv('plots/test_results.csv', index=False)
print("Test results saved to 'plots/test_results.csv'")